kernel : Python (Pixi)

In [ ]:
import anndata
import gc
import os
import pandas as pd
import scanpy as sc
from anndata import AnnData
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir

anndata.settings.allow_write_nullable_strings = True

h5ad_matrix_location = find_env_dir("H5AD_MATRIX")

# Loads 10x format data
def load_10x_data(mtx_file: str, barcodes_file: str, features_file: str, series: str) -> AnnData:
    adata: AnnData = sc.read_mtx(mtx_file).T

    with open(barcodes_file, "r") as f:
        barcodes = [line.strip().split("\t")[0] for line in f]
    with open(features_file, "r") as f:
        genes = [line.strip().split("\t")[0] for line in f]
    adata.obs_names = barcodes
    adata.var_names = genes

    # Converting to csr_matrix for memory efficiency and speed
    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)
    adata.obs["series"] = series
    return adata

In [ ]:
SERIES_NAME = "Drokhlyansky"
raw_count_matrix_location = os.path.join(find_env_dir("RAW_COUNT_MATRIX"), SERIES_NAME)

MTX_FILE = "gene_sorted-hli.matrix.mtx"
BARCODES_FILE = "hli.barcodes.tsv"
FEATURES_FILE = "hli.genes.tsv"
META_DATA_FILE = "all.meta.txt"
SAVE_FILE_NAME = SERIES_NAME + "_human.h5ad"

human_adata = load_10x_data(
    os.path.join(raw_count_matrix_location, MTX_FILE),
    os.path.join(raw_count_matrix_location, BARCODES_FILE),
    os.path.join(raw_count_matrix_location, FEATURES_FILE),
    series = SERIES_NAME + "_human"
)

meta = pd.read_csv(os.path.join(raw_count_matrix_location, META_DATA_FILE), sep="\t")
meta.set_index("NAME", inplace=True)
meta = meta[meta["Dataset"].fillna("").str.contains("all cells", case=False)]

assert isinstance(human_adata.obs, pd.DataFrame)
human_adata.obs = human_adata.obs.join(meta, how="left")

human_adata.obs.head() #type: ignore

/tmp/ipykernel_837455/1635604213.py:17: DtypeWarning: Columns (0: Age, 1: Location_ID, 2: Mouse_ID, 3: Patient_ID, 4: Segment, 5: Time, 6: Type, 7: nGene, 8: nUMI) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(os.path.join(raw_count_matrix_location, META_DATA_FILE), sep="\t")


,series,Age,Annotation,Dataset,Location_ID,Mouse_ID,Overload,Patient_ID,Region,Segment,Sex,Time,Type,Unique_ID,nGene,nUMI
H10_S46.GGAACTTCAAGAAGAG,Drokhlyansky_human,35,Fibroblast_2,Human colon all cells (10X),Sigmoid,NaN,1X,H10,Myenteric,NaN,M,NaN,NaN,S46,1166,1563
H10_S46.CCGTACTCACCTCGGA,Drokhlyansky_human,35,Epithelial,Human colon all cells (10X),Sigmoid,NaN,1X,H10,Myenteric,NaN,M,NaN,NaN,S46,1554,2061
H10_S46.GATGCTAAGAGCAATT,Drokhlyansky_human,35,Fibroblast_1,Human colon all cells (10X),Sigmoid,NaN,1X,H10,Myenteric,NaN,M,NaN,NaN,S46,1335,1760
H10_S46.CCCAGTTAGGGATCTG,Drokhlyansky_human,35,Myocyte_3,Human colon all cells (10X),Sigmoid,NaN,1X,H10,Myenteric,NaN,M,NaN,NaN,S46,1008,1367
H10_S46.GGATTACCAGGATTGG,Drokhlyansky_human,35,Myocyte_2,Human colon all cells (10X),Sigmoid,NaN,1X,H10,Myenteric,NaN,M,NaN,NaN,S46,1066,1398


In [ ]:
assert isinstance(human_adata.obs, pd.DataFrame)
human_adata.obs["age"] = human_adata.obs["Age"]
human_adata.obs["celltype"] = human_adata.obs["Annotation"]
human_adata.obs["location"] = human_adata.obs["Location_ID"]
human_adata.obs["overload"] = human_adata.obs["Overload"]
human_adata.obs["patient"] = human_adata.obs["Patient_ID"]
human_adata.obs["region"] = human_adata.obs["Region"]
human_adata.obs["sex"] = human_adata.obs["Sex"]
human_adata.obs["sample"] = human_adata.obs["Unique_ID"]

human_adata.obs["age"] = human_adata.obs["age"].astype(float)
keep = ["age", "celltype", "location", "overload", "patient", "region", "sex", "sample", "series", ]
human_adata.obs.drop(columns = [c for c in human_adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [27]:
human_adata.obs.index = human_adata.obs.index.astype(str)
human_adata.var.index = human_adata.var.index.astype(str)
human_adata.write_h5ad(os.path.join(h5ad_matrix_location, SAVE_FILE_NAME))
del human_adata
gc.collect()

4434

In [ ]:
SERIES_NAME = "Drokhlyansky"
raw_count_matrix_location = os.path.join(find_env_dir("RAW_COUNT_MATRIX"), SERIES_NAME)

MTX_FILE = "gene_sorted-mli.matrix.mtx"
BARCODES_FILE = "mli.barcodes.tsv"
FEATURES_FILE = "mli.genes.tsv"
META_DATA_FILE = "all.meta.txt"
SAVE_FILE_NAME = SERIES_NAME + "_mouse_colon.h5ad"

mouse_colon_adata = load_10x_data(
    os.path.join(raw_count_matrix_location, MTX_FILE),
    os.path.join(raw_count_matrix_location, BARCODES_FILE),
    os.path.join(raw_count_matrix_location, FEATURES_FILE),
    series = SERIES_NAME + "_mouse_colon"
)

meta = pd.read_csv(os.path.join(raw_count_matrix_location, META_DATA_FILE), sep="\t")
meta.set_index("NAME", inplace=True)
meta = meta[meta["Dataset"].fillna("").str.contains("all cells", case=False)]

assert isinstance(mouse_colon_adata.obs, pd.DataFrame)
mouse_colon_adata.obs = mouse_colon_adata.obs.join(meta, how="left")

mouse_colon_adata.obs.head() #type: ignore

/tmp/ipykernel_837455/3914783271.py:17: DtypeWarning: Columns (0: Age, 1: Location_ID, 2: Mouse_ID, 3: Patient_ID, 4: Segment, 5: Time, 6: Type, 7: nGene, 8: nUMI) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(os.path.join(raw_count_matrix_location, META_DATA_FILE), sep="\t")


,series,Age,Annotation,Dataset,Location_ID,Mouse_ID,Overload,Patient_ID,Region,Segment,Sex,Time,Type,Unique_ID,nGene,nUMI
M1_S1.AGTAGCTAGCCTGCCA,Drokhlyansky_mouse,NaN,Fibroblast,Mouse colon all cells (10X),NaN,M1,2x,NaN,colon,1.0,F,NaN,NaN,S1,1229,1634
M1_S1.CTGTGAATCAAGCCTA,Drokhlyansky_mouse,NaN,TA_2,Mouse colon all cells (10X),NaN,M1,2x,NaN,colon,1.0,F,NaN,NaN,S1,1159,1572
M1_S1.TGCTTGCTCGACACCG,Drokhlyansky_mouse,NaN,Glia,Mouse colon all cells (10X),NaN,M1,2x,NaN,colon,1.0,F,NaN,NaN,S1,2099,3385
M1_S1.TTCAATCTCATGTCTT,Drokhlyansky_mouse,NaN,Cycling_TA_2,Mouse colon all cells (10X),NaN,M1,2x,NaN,colon,1.0,F,NaN,NaN,S1,1651,2418
M1_S1.AGTCACATCTCCTGAC,Drokhlyansky_mouse,NaN,TA_1,Mouse colon all cells (10X),NaN,M1,2x,NaN,colon,1.0,F,NaN,NaN,S1,1577,2432


In [ ]:
assert isinstance(mouse_colon_adata.obs, pd.DataFrame)
mouse_colon_adata.obs["celltype"] = mouse_colon_adata.obs["Annotation"]
mouse_colon_adata.obs["mouse"] = mouse_colon_adata.obs["Mouse_ID"]
mouse_colon_adata.obs["overload"] = mouse_colon_adata.obs["Overload"]
mouse_colon_adata.obs["region"] = mouse_colon_adata.obs["Region"]
mouse_colon_adata.obs["sex"] = mouse_colon_adata.obs["Sex"]
mouse_colon_adata.obs["sample"] = mouse_colon_adata.obs["Unique_ID"]

keep = ["celltype", "mouse", "overload", "region", "sex", "sample", "series", ]
mouse_colon_adata.obs.drop(columns = [c for c in mouse_colon_adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [44]:
mouse_colon_adata.obs.index = mouse_colon_adata.obs.index.astype(str)
mouse_colon_adata.var.index = mouse_colon_adata.var.index.astype(str)
mouse_colon_adata.write_h5ad(os.path.join(h5ad_matrix_location, SAVE_FILE_NAME))
del mouse_colon_adata
gc.collect()

12126

In [45]:
SERIES_NAME = "Drokhlyansky"
raw_count_matrix_location = os.path.join(find_env_dir("RAW_COUNT_MATRIX"), SERIES_NAME)

MTX_FILE = "gene_sorted-msi.matrix.mtx"
BARCODES_FILE = "msi.barcodes.tsv"
FEATURES_FILE = "msi.genes.tsv"
META_DATA_FILE = "all.meta.txt"
SAVE_FILE_NAME = SERIES_NAME + "_mouse_ileum.h5ad"

mouse_ileum_adata = load_10x_data(
    os.path.join(raw_count_matrix_location, MTX_FILE),
    os.path.join(raw_count_matrix_location, BARCODES_FILE),
    os.path.join(raw_count_matrix_location, FEATURES_FILE),
    series = SERIES_NAME + "_mouse_ileum"
)

meta = pd.read_csv(os.path.join(raw_count_matrix_location, META_DATA_FILE), sep="\t")
meta.set_index("NAME", inplace=True)
meta = meta[meta["Dataset"].fillna("").str.contains("all cells", case=False)]

assert isinstance(mouse_ileum_adata.obs, pd.DataFrame)
mouse_ileum_adata.obs = mouse_ileum_adata.obs.join(meta, how="left")

mouse_ileum_adata.obs.head() #type: ignore

/tmp/ipykernel_837455/1190430292.py:17: DtypeWarning: Columns (0: Age, 1: Location_ID, 2: Mouse_ID, 3: Patient_ID, 4: Segment, 5: Time, 6: Type, 7: nGene, 8: nUMI) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(os.path.join(raw_count_matrix_location, META_DATA_FILE), sep="\t")


,series,Age,Annotation,Dataset,Location_ID,Mouse_ID,Overload,Patient_ID,Region,Segment,Sex,Time,Type,Unique_ID,nGene,nUMI
M1_S3.GTAGATCTCGCTACAA,Drokhlyansky_mouse_ileum,NaN,Paneth,Mouse ileum all cells (10X),NaN,M1,2x,NaN,ileum,NaN,F,NaN,NaN,S3,1123,1561
M1_S3.ATGACCACAGTCAGAG,Drokhlyansky_mouse_ileum,NaN,Enteroendocrine,Mouse ileum all cells (10X),NaN,M1,2x,NaN,ileum,NaN,F,NaN,NaN,S3,1689,2622
M1_S3.TCATGTTGTAACTAAG,Drokhlyansky_mouse_ileum,NaN,TA_5,Mouse ileum all cells (10X),NaN,M1,2x,NaN,ileum,NaN,F,NaN,NaN,S3,2258,4383
M1_S3.CATTGCCCAGAGATGC,Drokhlyansky_mouse_ileum,NaN,Paneth,Mouse ileum all cells (10X),NaN,M1,2x,NaN,ileum,NaN,F,NaN,NaN,S3,1723,2723
M1_S3.TTCCTAATCACTCGAA,Drokhlyansky_mouse_ileum,NaN,TA_2,Mouse ileum all cells (10X),NaN,M1,2x,NaN,ileum,NaN,F,NaN,NaN,S3,1604,2436


In [ ]:
assert isinstance(mouse_ileum_adata.obs, pd.DataFrame)
mouse_ileum_adata.obs["celltype"] = mouse_ileum_adata.obs["Annotation"]
mouse_ileum_adata.obs["mouse"] = mouse_ileum_adata.obs["Mouse_ID"]
mouse_ileum_adata.obs["overload"] = mouse_ileum_adata.obs["Overload"]
mouse_ileum_adata.obs["region"] = mouse_ileum_adata.obs["Region"]
mouse_ileum_adata.obs["sex"] = mouse_ileum_adata.obs["Sex"]
mouse_ileum_adata.obs["sample"] = mouse_ileum_adata.obs["Unique_ID"]

keep = ["celltype", "mouse", "overload", "region", "sex", "sample", "series", ]
mouse_ileum_adata.obs.drop(columns = [c for c in mouse_ileum_adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [47]:
mouse_ileum_adata.obs.index = mouse_ileum_adata.obs.index.astype(str)
mouse_ileum_adata.var.index = mouse_ileum_adata.var.index.astype(str)
mouse_ileum_adata.write_h5ad(os.path.join(h5ad_matrix_location, SAVE_FILE_NAME))
del mouse_ileum_adata
gc.collect()

0